# Preempting Hard Drive Failures: EDA & Preprocessing
Backblaze Drive Stats, Q1 + Q2 2026 (Jan 1 to Jun 30).

# 0. Get the data
The raw data is too big to submit (30 MB limit), so this cell downloads any missing quarter from Backblaze and unpacks its daily CSVs into `../data/raw`. The first run downloads about 3.4 GB and needs about 24 GB of disk; later runs skip it.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve, urlcleanup
from zipfile import ZipFile

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)
for q, first_day in {"Q1": "2026-01-01", "Q2": "2026-04-01"}.items():
    if not (RAW / f"{first_day}.csv").exists():
        zip_path, _ = urlretrieve(f"https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/data_{q}_2026.zip")
        with ZipFile(zip_path) as z:
            for name in z.namelist():
                if name.endswith(".csv") and "__MACOSX" not in name:
                    (RAW / Path(name).name).write_bytes(z.read(name))
        urlcleanup()

len(list(RAW.glob("*.csv")))

181

# 1. Overview
Each row is one drive on one day. We count rows, columns, drives, the date range and failures over all daily files, show a few raw records, and rank drive models by failures. We keep the 3 models with the most failures: each model reports a different set of SMART attributes, so mixing many models would leave most columns empty.

In [2]:
con = duckdb.connect()
CSV = f"read_csv('{RAW}/*.csv', all_varchar=true, union_by_name=true)"

con.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT serial_number)     AS drives,
        MIN(date)                         AS first_day,
        MAX(date)                         AS last_day,
        SUM(CAST(failure AS INTEGER))     AS failure_rows
    FROM {CSV}
""").df().assign(columns=len(con.sql(f"SELECT * FROM {CSV} LIMIT 0").columns))

,total_rows,drives,first_day,last_day,failure_rows,columns
0,62565487,363548,2026-01-01,2026-06-30,2552.0,197


In [3]:
con.sql(f"SELECT * FROM {CSV} LIMIT 5").df()

,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2026-01-01,000a43e7dee60010,DELLBOSS VD,480036847616,0,ams5,031,2017,05,None,...,None,None,None,None,None,None,None,None,None,None
1,2026-01-01,00ac486b9bec0010,DELLBOSS VD,480036847616,0,ams5,031,2013,16,None,...,None,None,None,None,None,None,None,None,None,None
2,2026-01-01,01b776caeb6a0010,DELLBOSS VD,480036847616,0,ams5,031,2014,11,None,...,None,None,None,None,None,None,None,None,None,None
3,2026-01-01,02308a3415be0010,DELLBOSS VD,480036847616,0,ams5,031,2007,04,None,...,None,None,None,None,None,None,None,None,None,None
4,2026-01-01,0394cd7bc5d00010,DELLBOSS VD,480036847616,0,ams5,031,2017,04,None,...,None,None,None,None,None,None,None,None,None,None


In [4]:
models = con.sql(f"""
    SELECT model,
           COUNT(DISTINCT serial_number)  AS drives,
           SUM(CAST(failure AS INTEGER))  AS failures,
           ROUND(100.0 * failures / drives, 2) AS pct_failed
    FROM {CSV}
    GROUP BY model
    ORDER BY failures DESC
    LIMIT 10
""").df()

MODELS = list(models.model[:3])
models

,model,drives,failures,pct_failed
0,HGST HUH721212ALN604,9790,275.0,2.81
1,ST12000NM0008,18652,274.0,1.47
2,TOSHIBA MG07ACA14TA,37265,273.0,0.73
3,TOSHIBA MG08ACA16TA,40036,204.0,0.51
4,HGST HUH721212ALE604,13283,196.0,1.48
5,ST24000NM002H,9739,156.0,1.60
6,ST8000NM0055,13241,151.0,1.14
7,WDC WUH721816ALE6L4,26349,130.0,0.49
8,TOSHIBA MG10ACA20TE,25314,106.0,0.42
9,WDC WUH722222ALE6L4,45707,105.0,0.23


# 2. Uniform drive sample
The 3 models still have millions of rows, too many to plot in memory. We keep about 20% of drives, picked by a hash of the serial number. The same rule applies to failed and healthy drives, so the natural class imbalance is unchanged (compare `pct_failed` below), and each kept drive keeps its full history. Duplicate (drive, day) rows are dropped here, and the text columns are cast to proper types.

In [5]:
in_list = ", ".join(f"'{m}'" for m in MODELS)

con.sql(f"""
    COPY (
        SELECT
            CAST(date AS DATE)                 AS date,
            serial_number,
            model,
            TRY_CAST(capacity_bytes AS BIGINT) AS capacity_bytes,
            CAST(failure AS INTEGER)           AS failure,
            datacenter, cluster_id, vault_id, pod_id, pod_slot_num,
            TRY_CAST(COLUMNS('^smart_') AS DOUBLE)
        FROM {CSV}
        WHERE model IN ({in_list})
          AND hash(serial_number) % 100 < 20
        QUALIFY row_number() OVER (PARTITION BY serial_number, date) = 1
        ORDER BY serial_number, date
    ) TO '../data/sample.parquet' (FORMAT PARQUET)
""")

con.sql("DESCRIBE '../data/sample.parquet'").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,serial_number,VARCHAR,YES,None,None,None
2,model,VARCHAR,YES,None,None,None
3,capacity_bytes,BIGINT,YES,None,None,None
4,failure,INTEGER,YES,None,None,None
...,...,...,...,...,...,...
191,smart_252_raw,DOUBLE,YES,None,None,None
192,smart_254_normalized,DOUBLE,YES,None,None,None
193,smart_254_raw,DOUBLE,YES,None,None,None
194,smart_255_normalized,DOUBLE,YES,None,None,None


In [6]:
sample = con.sql("""
    SELECT model,
           COUNT(*)                       AS rows,
           COUNT(DISTINCT serial_number)  AS drives,
           SUM(failure)                   AS failures,
           ROUND(100.0 * failures / drives, 2) AS pct_failed
    FROM '../data/sample.parquet'
    GROUP BY model
""").df()

sample.merge(models[["model", "pct_failed"]], on="model", suffixes=("", "_all"))

,model,rows,drives,failures,pct_failed,pct_failed_all
0,ST12000NM0008,666385,3725,55.0,1.48,1.47
1,TOSHIBA MG07ACA14TA,1342435,7455,56.0,0.75,0.73
2,HGST HUH721212ALN604,352823,1996,58.0,2.91,2.81


# 3. Data quality
**Duplicates:** we count (drive, day) pairs that appear more than once in the raw data. Step 2 already kept only one row for each.

**Missing values:** the table shows how often each column is filled, per model (100 = always). Columns a model never reports are 0. Columns that are empty for all 3 models (`empty`) carry no information and are dropped in step 4. The other gaps are structural (that model doesn't report that attribute), so they stay as NaN for now and are imputed during preprocessing.

In [7]:
con.sql(f"""
    SELECT COUNT(*) AS duplicated_drive_days
    FROM (SELECT serial_number, date FROM {CSV} GROUP BY ALL HAVING COUNT(*) > 1)
""").df()

,duplicated_drive_days
0,0


In [8]:
fill = con.sql("""
    SELECT model, COUNT(*) AS n_rows, COUNT(COLUMNS(* EXCLUDE (model)))
    FROM '../data/sample.parquet'
    GROUP BY model
""").df().set_index("model")

pct = (fill.drop(columns="n_rows").div(fill["n_rows"], axis=0) * 100).round(0)
empty = list(pct.columns[(pct == 0).all()])
print(f"{len(empty)} empty columns dropped in step 4")
pct.T[(pct.T > 0).any(axis=1)]

116 empty columns dropped in step 4


model,HGST HUH721212ALN604,ST12000NM0008,TOSHIBA MG07ACA14TA
date,100.0,100.0,100.0
serial_number,100.0,100.0,100.0
capacity_bytes,100.0,100.0,100.0
failure,100.0,100.0,100.0
datacenter,100.0,100.0,100.0
...,...,...,...
smart_240_raw,0.0,100.0,100.0
smart_241_normalized,1.0,100.0,0.0
smart_241_raw,1.0,100.0,0.0
smart_242_normalized,1.0,100.0,0.0


# 4. Prediction labels
For each horizon h (3, 7 and 30 days), `label_{h}d` is 1 if the drive fails within the next h days, otherwise 0. It is NULL (unknown) when:
- the row comes after the drive's failure, or
- the drive hasn't failed and its next h days run past the last day of data, so we can't tell yet.

`days_to_failure` is kept for later steps. The asserts check that drives that never fail get no positive labels, and that each failed drive has at most h positive days.

In [9]:
HORIZONS = [3, 7, 30]

labels_sql = ",\n               ".join(f"""CASE WHEN days_to_failure < 0 THEN NULL
                   WHEN days_to_failure < {h} THEN 1
                   WHEN days_to_failure IS NOT NULL THEN 0
                   WHEN date > last_day - {h - 1} THEN NULL
                   ELSE 0 END AS label_{h}d""" for h in HORIZONS)

con.sql(f"""
    COPY (
        SELECT * EXCLUDE ({", ".join(["last_day"] + empty)}),
               {labels_sql}
        FROM (
            SELECT *,
                   date_diff('day', date, MIN(CASE WHEN failure = 1 THEN date END)
                                          OVER (PARTITION BY serial_number)) AS days_to_failure,
                   MAX(date) OVER () AS last_day
            FROM '../data/sample.parquet'
        )
        ORDER BY serial_number, date
    ) TO '../data/labeled.parquet' (FORMAT PARQUET)
""")

per_drive = con.sql("""
    SELECT MAX(failure) AS failed, SUM(COLUMNS('^label_'))
    FROM '../data/labeled.parquet'
    GROUP BY serial_number
""").df().fillna(0)
assert (per_drive.loc[per_drive.failed == 0, ["label_3d", "label_7d", "label_30d"]] == 0).all(axis=None)
assert all((per_drive[f"label_{h}d"] <= h).all() for h in HORIZONS)

con.sql("""
    SELECT month(date) AS month, COUNT(*) AS rows, SUM(COLUMNS('^label_'))
    FROM '../data/labeled.parquet'
    GROUP BY month
    ORDER BY month
""").df()

,month,rows,label_3d,label_7d,label_30d
0,1,406525,53.0,117.0,551.0
1,2,366890,45.0,105.0,536.0
2,3,405257,69.0,160.0,735.0
3,4,391578,109.0,244.0,1129.0
4,5,403005,88.0,178.0,707.0
5,6,388388,58.0,128.0,281.0
